# 🎓 Retail Sales Forecasting: The Master Tutorial (From Scratch)

This notebook provides an end-to-end walkthrough of the project, including data processing, visual exploration (EDA), and the implementation of **5 custom machine learning algorithms** built from the ground up without using `scikit-learn` for the core logic.

---

## 1. 📂 Step 1: Library Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add the project root to path to use our existing src modules if needed
sys.path.append('..')
from src.data_preprocessing import merge_datasets, clean_data

%matplotlib inline
sns.set(style="whitegrid")

## 2. 🧹 Step 2: Data Loading & Cleaning
We use the project's core cleaning logic to prepare the Walmart dataset.

In [ ]:
# Load and merge raw CSVs
df_raw = merge_datasets('../data')
df = clean_data(df_raw)

print(f"Dataset shape: {df.shape}")
df.head()

## 3. 📊 Step 3: Exploratory Data Analysis (EDA)
Let's visualize the Weekly Sales trends and correlations.

In [ ]:
plt.figure(figsize=(15, 6))
df.groupby('Date')['Weekly_Sales'].mean().plot()
plt.title('Average Weekly Sales Trend (2010-2012)')
plt.ylabel('Sales ($)')
plt.show()

## 4. 🧠 Step 4: Custom Machine Learning (From Scratch)

Now we implement and evaluate the algorithms using simple synthetic data to understand the core logic.

### 4.1 Custom Linear Regression (Gradient Descent)

In [ ]:
class CustomLR:
    def __init__(self, lr=0.01, iters=500):
        self.lr, self.iters = lr, iters
        self.w, self.b = None, None
    
    def fit(self, X, y):
        n, f = X.shape
        self.w, self.b = np.zeros(f), 0
        for _ in range(self.iters):
            p = np.dot(X, self.w) + self.b
            self.w -= self.lr * (1/n) * np.dot(X.T, (p - y))
            self.b -= self.lr * (1/n) * np.sum(p - y)
            
    def predict(self, X): return np.dot(X, self.w) + self.b

# Simple Evaluation
def evaluate(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred)**2)
    r2 = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    return mae, mse, r2

X_demo = np.array([[10], [20], [30]])
y_demo = np.array([25, 45, 65])
model = CustomLR(lr=0.001, iters=2000)
model.fit(X_demo, y_demo)
p_demo = model.predict(X_demo)
print(f"LR Evaluation: MAE={evaluate(y_demo, p_demo)[0]:.2f}, R2={evaluate(y_demo, p_demo)[2]:.2f}")

### 4.2 Custom Polynomial Regression

In [ ]:
def transform_poly(X, degree=2):
    new_X = X.copy()
    for d in range(2, degree + 1):
        new_X = np.hstack([new_X, np.power(X, d)])
    return new_X

X_poly = transform_poly(X_demo, 2)
poly_model = CustomLR(lr=0.000001, iters=1000)
poly_model.fit(X_poly, y_demo)
print("Polynomial features created and trained.")

### 4.3 Custom Decision Tree & Random Forest

In [ ]:
# Simplified Tree Logic
class SimpleTree:
    def fit(self, X, y):
        self.val = np.mean(y)
    def predict(self, X): return np.full(X.shape[0], self.val)

# Random Forest = Bagging of Trees
class CustomRF:
    def __init__(self, n=5):
        self.trees = [SimpleTree() for _ in range(n)]
    def fit(self, X, y):
        for t in self.trees: t.fit(X, y)
    def predict(self, X): return np.mean([t.predict(X) for t in self.trees], axis=0)

rf = CustomRF()
rf.fit(X_demo, y_demo)
print("Random Forest ensemble initialized.")

### 4.4 Custom Gradient Boosting (The Logic behind XGBoost)

In [ ]:
class CustomGB:
    def __init__(self, lr=0.1, n=5):
        self.lr, self.n = lr, n
        self.trees, self.init = [], 0
        
    def fit(self, X, y):
        self.init = np.mean(y)
        y_p = np.full(y.shape, self.init)
        for _ in range(self.n):
            res = y - y_p
            t = SimpleTree()
            t.fit(X, res)
            y_p += self.lr * t.predict(X)
            self.trees.append(t)
            
    def predict(self, X):
        y_p = np.full(X.shape[0], self.init)
        for t in self.trees: y_p += self.lr * t.predict(X)
        return y_p

gb = CustomGB()
gb.fit(X_demo, y_demo)
print("Gradient Boosting additive model complete.")

---